# Outlier Detection and Data Quality

In [24]:
import pandas as pd

In [25]:
sold = pd.read_csv("sold_with_metrics.csv")

C:\Users\mukun\AppData\Local\Temp\ipykernel_11932\455216933.py:1: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  sold = pd.read_csv("sold_with_metrics.csv")


### Calculating IQR and Removing Records that Fall Outside a Defined Statistical Range

In this section, I will remove outliers for key fields(Close Price, Living Area, Days on Market) because it can skew median prices and misrepresent the typical market. After removing outliers for each field, I will compare the updated lengths and medians of the dataset to see how removing outliers affected the dataset. I will save both the full flagged dataset before filtering for outliers and the dataset with outliers removed.

##### Close Price Outliers

In [26]:
len(sold)

60371

In [27]:
sold["ClosePrice"].median()

np.float64(825000.0)

In [28]:
Q1 = sold['ClosePrice'].quantile(0.25)
Q3 = sold['ClosePrice'].quantile(0.75)
IQR = Q3 - Q1
lower_close = Q1 - 1.5 * IQR
upper_close = Q3 + 1.5 * IQR
sold_new = sold[(sold['ClosePrice'] >= lower_close) & (sold['ClosePrice'] <= upper_close)]

In [29]:
len(sold_new)

55829

In [30]:
sold_new["ClosePrice"].median()

np.float64(785000.0)

##### Living Area Outliers

In [31]:
sold_new["LivingArea"].median()

np.float64(1607.0)

In [32]:
Q1 = sold['LivingArea'].quantile(0.25)
Q3 = sold['LivingArea'].quantile(0.75)
IQR = Q3 - Q1
lower_area = Q1 - 1.5 * IQR
upper_area = Q3 + 1.5 * IQR
sold_new = sold_new[(sold_new['LivingArea'] >= lower_area) & (sold_new['LivingArea'] <= upper_area)]

In [33]:
len(sold_new)

54917

In [34]:
sold_new["LivingArea"].median()

np.float64(1595.0)

##### Days on Market Outliers

In [35]:
sold_new["DaysOnMarket"].median()

np.float64(18.0)

In [36]:
Q1 = sold['DaysOnMarket'].quantile(0.25)
Q3 = sold['DaysOnMarket'].quantile(0.75)
IQR = Q3 - Q1
lower_days = Q1 - 1.5 * IQR
upper_days = Q3 + 1.5 * IQR
sold_new = sold_new[(sold_new['DaysOnMarket'] >= lower_days) & (sold_new['DaysOnMarket'] <= upper_days)]

In [37]:
len(sold_new)

50214

In [38]:
sold_new["DaysOnMarket"].median()

np.float64(15.0)

### Updating Original Dataset by Flagging it For Outliers for Any Metric

In [39]:
sold["OutlierDetection"] = ((sold['ClosePrice'] < lower_close) | (sold['ClosePrice'] > upper_close) | 
                            (sold['LivingArea'] < lower_area) | (sold['LivingArea'] > upper_area) | 
                            (sold['DaysOnMarket'] < lower_days) | (sold['DaysOnMarket'] > upper_days))

##### Valiidating that the Filtered Dataset and Flagged Dataset both Identify Outliers Accurately

In [40]:
len(sold) - len(sold_new)

10157

In [41]:
len(sold[sold["OutlierDetection"] == True])

10148

Although there is a difference of 9 observations between the filtered and flagged dataset's identification of outliers, this discrepancy is minimal relative to the size of the dataset and does not affect the validity of the results(is likely due to a subtle difference in calculation of outliers in the two datasets). The validation checks confirm that the filtering and outlier detection processes are operating as expected, correctly distinguishing between retained and flagged observations.

### Conclusion

Overall, through using the IQR method, about 10,000 listings were considered outliers in one or more of the metrics I used. The medians also consistently decreased across all used metrics after filtering for outliers.

### Saving both Full Flagged Dataset and Outlier Filtered Dataset as CSVs

In [42]:
sold_new.to_csv("final_sold_without_outliers.csv", index=False)
sold.to_csv("final_flagged_sold.csv", index=False)